# Select background subtracted files: cell ch

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.image as mpimg
import sys
import shutil
src_path = str(Path.cwd().parent.parent)
if src_path not in sys.path:
    sys.path.append(src_path)
from src.d01_init_proc import vis_and_rescale
from src.d00_utils import dirnames as dn
from src.d00_utils import utilities as utils
from src.d01_init_proc.subtractbg import subtract_bg

## Load background subtract dataframe

In [ ]:
ch_content = 'cell'
ch = 1
bgsb_by_mask = True

In [ ]:
bgsub_df_path = Path(input())

In [ ]:
bgsub_df = pd.read_csv(bgsub_df_path)
bgsub_df['basename'] = [name[:(np.char.find(name, 'aligned') + len('aligned'))] for name in bgsub_df['input image name']]
print(len(bgsub_df))
bgsub_df.head()

## Create or load table with mask information

In [ ]:
proc_dirpath = utils.get_proc_dirpath(bgsub_df_path)
tables_dirpath = proc_dirpath / dn.tables_dirname

In [ ]:
masks_df_path = tables_dirpath / 'masks_df.csv'

if masks_df_path.is_file():
    masks_df = pd.read_csv(masks_df_path)
else:
    polygonmasks_dirpath = proc_dirpath / dn.masks_dirname / dn.polygon_ROI_dirname
    assert polygonmasks_dirpath.is_dir()

    masknames = [mask.name for mask in polygonmasks_dirpath.glob('*.png')]
    masknames.sort()

    masks_df = pd.DataFrame()
    masks_df['mask name'] = masknames
    masks_df['mask dirpath'] = polygonmasks_dirpath
    splits = masks_df['mask name'].str.split('_')

    # may need to modify based on names
    masks_df['basename'] = [name[:len(name)-9] for name in masks_df['mask name']]
    masks_df['wellID'] = splits.str[2]
    masks_df['scene'] = (splits.str[3]).str.split('-').str[0]
    masks_df['ROI'] = [name[len(name)-8:len(name)-4] for name in masks_df['mask name']]
    masks_df.head()

    masks_df_path = bgsub_df_path.parent / 'masks_df.csv'
    utils.safe_save_csv(masks_df, masks_df_path)

In [ ]:
bgsub_masks_df_path = bgsub_df_path.parent / 'bgsub_cell_masks.csv'

if bgsb_by_mask is True:
    bgsub_df['ROI'] = bgsub_df['input image name'].str.split('.ome.tif').str[0].str.split('_').str[-1]
    masks_df['basename_roi'] = masks_df['basename'] + '_' + masks_df['ROI']
    bgsub_df['basename_roi'] = bgsub_df['basename'] + '_' + bgsub_df['ROI']
    bgsub_cell_masks_df = pd.merge(masks_df, bgsub_df, on='basename_roi', how='left', suffixes = ['', '_y'])
else:
    bgsub_cell_masks_df = pd.merge(masks_df, bgsub_df, on='basename', how='left', suffixes = ['', '_y'])

assert (len(masks_df) == (len(bgsub_cell_masks_df)))

bgsub_cell_masks_df.head()

In [ ]:
cols = np.array(bgsub_cell_masks_df.columns.to_list())

for i, col in enumerate(cols):
    print(f'{i}: {col}')

In [ ]:
new_col_order = ['img idx', 'basename', 'wellID', 'scene', 'ROI', 'selected param', 'notes', 'BG subtract?', 'ch_to_process', 'outlier_percentiles', 'rescale_perc_grayval', 'sigmas_smoothing', '# prev tested params', 'mask name', 'mask dirpath', 'input image name', 'input dirname', 'bgsb dirpath', 'bin dirpath']

bgsub_cell_masks_df = bgsub_cell_masks_df[new_col_order]
bgsub_cell_masks_df.head()

In [ ]:
utils.safe_save_csv(bgsub_cell_masks_df, bgsub_masks_df_path)

## Visualize background subtracted figures

In [ ]:
bin_dirpath = Path(bgsub_cell_masks_df.at[0, 'bin dirpath'])
fig_dirpath = bin_dirpath / dn.figs_dirname
assert fig_dirpath.is_dir()

In [ ]:
figpaths = list(fig_dirpath.glob('*.png'))
figpaths.sort()
num_figs = len(figpaths)
print(num_figs)

In [ ]:
subset=None

#### Note: open "bgsub" file and use visualized figures to fill in the "selected params" column

In [ ]:
subset = None
vis_and_rescale.show_figs(fig_dirpath, subset=subset)

## Re-run background subtraction (optional)

In [ ]:
bgsub_masks_df = pd.read_csv(bgsub_masks_df_path)
bgsub_masks_df.head()

In [ ]:
input_dirpath = utils.get_proc_dirpath(polygonmasks_dirpath) / 'caax_cell_stack'
#input_dirpath = utils.get_proc_dirpath(polygonmasks_dirpath) / dn.stack_dirname

subtract_bg(input_dirpath, bgsub_masks_df_path)

## Update bgsub masks dataframe with selected images

In [ ]:
bgsub_masks_df_path

In [ ]:
bgsub_masks_df = pd.read_csv(bgsub_masks_df_path)
bgsub_masks_df.head()

In [ ]:
bgsub_masks_df[f'bin {ch_content} img']  = bgsub_masks_df['input image name'].str.split('.').str[0] + '_p' + bgsub_masks_df['selected param'].astype('string') + '.ome.tif'
bgsub_masks_df[f'bin {ch_content} img'].loc[bgsub_masks_df['selected param']=='omit'] = 'NA'

bgsub_masks_df.head()

In [ ]:
sel_bin_dirpath = bin_dirpath.parent / (bin_dirpath.name.split('_init')[0] + '_sel')
print(sel_bin_dirpath)

In [ ]:
# record selected background-subtracted image dirpath
bgsub_masks_df[f'bin {ch_content} dirpath'] = sel_bin_dirpath

In [ ]:
utils.safe_save_csv(bgsub_masks_df, bgsub_masks_df_path)

# Create dataframe with masks and corresponding selected bgsub images 

In [ ]:
sel_bgsub_df_path = tables_dirpath / 'bgsub_selected_bymask.csv'

if sel_bgsub_df_path.is_file():
    sel_bgsub_df = pd.read_csv(sel_bgsub_df_path)
else:
    sel_bgsub_df = masks_df

print(f'pre-merge sel bgsub df rows: {len(sel_bgsub_df)}')    
print(f'bgsub masks df: {len(bgsub_masks_df)}')
sel_bgsub_df = pd.merge(sel_bgsub_df, bgsub_masks_df[['mask name', f'bin {ch_content} img', f'bin {ch_content} dirpath']])
sel_bgsub_df[f'bin {ch_content} notes'] = bgsub_df['notes']
print(f'sel_bgsub_df rows: {len(sel_bgsub_df)}')
    
sel_bgsub_df


In [ ]:
tables_dirpath = utils.get_proc_dirpath(bgsub_df_path) / dn.tables_dirname
sel_bgsub_df_path = tables_dirpath / 'bgsub_selected_bymask.csv'
utils.safe_save_csv(sel_bgsub_df, sel_bgsub_df_path)

## Create thresholds table with only selected images

In [ ]:
bgsub_thresh_path = str(bgsub_df_path).split('.csv')[0] + '_thresh.csv'
bgsub_thresh_df = pd.read_csv(bgsub_thresh_path)
bgsub_thresh_df.head()

In [ ]:
bgsub_thresh_df['image name'] = bgsub_thresh_df['image name'] + '.ome.tif'

In [ ]:
sel_imgs_uniq = [img for img in sel_bgsub_df[f'bin {ch_content} img'].unique() if img != 'NA']
print(sel_imgs_uniq)
#bgsub_thresh_sel_df = bgsub_thresh_df[bgsub_thresh_df['image name'].str.contains('|'.join(sel_imgs_uniq))]
#bgsub_thresh_sel_df.head()

In [ ]:
print(len(bgsub_thresh_sel_df))

In [ ]:
bgsub_thresh_sel_path = Path(str(bgsub_df_path).split('.csv')[0] + '_thresh_selected.csv')
print(bgsub_thresh_sel_path.name)

In [ ]:
utils.safe_save_csv(bgsub_thresh_sel_df, bgsub_thresh_sel_path)

## Move selected images

In [ ]:
assert bin_dirpath.is_dir()
print(f'bin dirpath: {bin_dirpath}\n')
print(f'sel bin dirpath: {sel_bin_dirpath}\n')

In [ ]:
sel_fig_dirpath.mkdir(parents=True, exist_ok=True)

In [ ]:
imgs_moved = 0

for img in sel_imgs_uniq:
    # move bg subtracted files
    imgpath = bin_dirpath / img

    if imgpath.is_file():
        shutil.copy(imgpath, sel_bin_dirpath / img)
        imgs_moved = imgs_moved + 1
    else:
        print(f'{img} not found.') 
        
print(f'Copied {imgs_moved} images.')